# 🐍 Python from Scratch — Module 7: Modules, Packages, pip, and venv

### How Python organizes code — and how to use someone else's

Every notebook so far has been one self-contained whole. In practice, programs are
split across many files, and you almost never write everything from scratch — you rely
on existing libraries. This module ties it all together: `import`, your own modules,
`pip`, and virtual environments.

## Table of Contents

1. [Why split code into modules](#sec1)
2. [`import` — the standard library](#sec2)
3. [Creating your own module](#sec3)
4. [`if __name__ == "__main__":`](#sec4)
5. [Packages — modules of modules](#sec5)
6. [`pip` and third-party libraries](#sec6)
7. [Virtual environments (venv)](#sec7)
8. [A quick tour of useful modules](#sec8)
9. [Fun fact: PyPI and "batteries included"](#sec9)
10. [Module summary](#sec10)
11. [Exercises](#sec11)

---

<a id="sec1"></a>
## 1. Why split code into modules

Back in Module 4, the DRY principle said: don't repeat code **within a single file**.
The same principle scales up: don't repeat code **across projects**. The BMI-calculating
function you wrote in Module 1 would be handy in another project too. Instead of
copy-pasting it every time, you can save it once in a separate file (a **module**) and
import it whenever you need it.

<a id="sec2"></a>
## 2. `import` — the standard library

Python ships with a huge **standard library** — modules available right away, with no
installation needed. `import` makes a module's contents available in your code, in a
few different ways.

In [ ]:
import math
print(math.sqrt(16))     # access through the module name
print(math.pi)

import math as m          # alias - a shorter name
print(m.ceil(4.1))        # round up
print(m.floor(4.9))       # round down

from math import sqrt, pi   # import specific names - no math. prefix needed
print(sqrt(25))
print(pi)

# from math import *   # imports EVERYTHING - see the warning below

> ⚠️ **Avoid `from module import *`**
>
> Importing everything at once clutters your namespace and makes it hard to trace where a name came from - if two modules define a function with the same name, one silently overwrites the other. It's better to import specific names, or the whole module under its (possibly shortened) own name.

<a id="sec3"></a>
## 3. Creating your own module

A module is just a `.py` file with function/class/variable definitions. In Jupyter, you
can create one right from a cell with the `%%writefile` magic command (it must be the
first line of the cell) — it saves the rest of the cell's contents to the given file.

In [ ]:
%%writefile my_module.py
"""A simple module with a couple of helper functions."""

def greet(name):
    return f"Hi, {name}! This is a greeting from the module."

def double(x):
    return x * 2

In [ ]:
import my_module

print(my_module.greet("Kamil"))
print(my_module.double(21))

> 💡 **Fun fact**
>
> `%%writefile` is a so-called *magic command* from IPython/Jupyter (hence the `%%`) - it isn't part of Python itself, it only works inside a notebook. Outside a notebook, you'd just create a `.py` file in a code editor and that file is the module - `%%writefile` only simulates that step, so it can be shown in a single notebook file.

<a id="sec4"></a>
## 4. `if __name__ == "__main__":`

Every module has a built-in `__name__` variable. When a file is **run directly**
(e.g. `python my_module.py`), `__name__` is set to `"__main__"`. When the file is
**imported** from another file, `__name__` is set to the module's name instead. This
idiom lets you put code in a file that only runs when it's executed directly, not when
it's imported.

In [ ]:
%%writefile conversions.py
"""A module with temperature conversion functions."""

def celsius_to_fahrenheit(c):
    return c * 9 / 5 + 32

def fahrenheit_to_celsius(f):
    return (f - 32) * 5 / 9

if __name__ == "__main__":
    # This block runs ONLY with "python conversions.py",
    # it does NOT run with "import conversions"
    print("Running conversions.py directly - demo mode active")
    print(celsius_to_fahrenheit(100))

In [ ]:
import conversions
print(conversions.celsius_to_fahrenheit(20))
# Notice: the import did NOT print "Running conversions.py directly..."

# Now let's run the file directly as a script (! runs a terminal command):
!python conversions.py

> 💡 **Where this is handy**
>
> Thanks to this idiom, the same file can be **both** a module to import (in another program) **and** a standalone script with a demo/test of its own functions, runnable straight from a terminal - a very common pattern in real Python projects.

<a id="sec5"></a>
## 5. Packages — modules of modules

A **package** is a folder containing several modules, plus an `__init__.py` file (which
can be empty) that tells Python "this is a package, not just a regular folder". Packages
let you group related modules together, e.g. `tools/text.py` and `tools/numbers.py`.

In [ ]:
import os
os.makedirs("my_package", exist_ok=True)

In [ ]:
%%writefile my_package/__init__.py
# This file (which can be empty) tells Python that "my_package" is a package

In [ ]:
%%writefile my_package/text.py
def reverse(text):
    return text[::-1]

In [ ]:
from my_package import text
print(text.reverse("Python"))

# Or import a specific function directly:
from my_package.text import reverse
print(reverse("Kamil"))

<a id="sec6"></a>
## 6. `pip` and third-party libraries

The standard library (like `math` or `random`) is built in. But the Python world also
has hundreds of thousands of **third-party packages** published on PyPI (the *Python
Package Index*) — you install them with `pip install package_name`, typed into a
terminal (not into a Python cell!).

In [ ]:
# A "!" at the start of a line in Jupyter runs a terminal command, not Python:
!pip --version
!pip list

> 💡 **`requirements.txt`**
>
> So that someone else (or you, six months from now) can recreate a project's exact same dependencies, they get saved in a `requirements.txt` file (one library per line, e.g. `pandas==2.2.0`). It's generated with `pip freeze > requirements.txt`, and everything in it is installed at once with `pip install -r requirements.txt`.

<a id="sec7"></a>
## 7. Virtual environments (venv)

The problem: project A needs `pandas==1.5`, while project B on the same computer needs
`pandas==2.2`. If libraries installed globally, one of the projects would break. A
**virtual environment** is an isolated, independent "copy" of Python with its own set of
installed packages — each project gets its own.

These commands run **in a terminal**, not in a notebook cell:

```bash
python -m venv env          # creates a new environment in a folder called "env"

# activate on Windows:
env\Scripts\activate

# activate on macOS / Linux:
source env/bin/activate

pip install -r requirements.txt   # installs dependencies ONLY into this environment

deactivate                   # leave the virtual environment
```

> ⚠️ **Anaconda has its own equivalent**
>
> Since you're using Jupyter through Anaconda - Anaconda has its own environment system, `conda create -n my_env python=3.11` and `conda activate my_env`, which works on the same principle as `venv`, just with a different command. You don't need to use both at once - either one is enough.

> 💡 **Fun fact**
>
> The problem described above has a name: «dependency hell» - a situation where different projects (or different libraries within the same project) require incompatible versions of the same dependency. Virtual environments don't solve this 100%, but they drastically shrink the scale of the problem.

<a id="sec8"></a>
## 8. A quick tour of useful modules

A few standard-library modules you'll likely need sooner or later:

| Module | What it's for |
|---|---|
| `math` | math functions (`sqrt`, `pi`, `ceil`, `floor`...) |
| `random` | random numbers, random choice, shuffling |
| `datetime` | dates and times, date arithmetic |
| `os` | file system, path, and folder operations |
| `sys` | interacting with the Python interpreter (arguments, paths) |
| `json` | reading and writing data in JSON format |

In [ ]:
from datetime import datetime, timedelta

now = datetime.now()
print(now)

in_a_week = now + timedelta(days=7)
print(in_a_week)

import json

data = {"name": "Kamil", "languages": ["Python", "SQL"]}
with open("data.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2)

with open("data.json", "r", encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded, type(loaded))

<a id="sec9"></a>
## 9. Fun fact: PyPI and "batteries included"

Python's philosophy from the start has been "batteries included" — the standard library
is deliberately rich, so basic tasks (working with files, dates, text, JSON) never
require any installation. And yet PyPI still counts **over 500,000 packages** — from
`requests` (HTTP requests) and `pandas` (data analysis), to libraries for literally
everything, from games to spaceflight (literally — NASA publishes packages on PyPI).

<a id="sec10"></a>
## 10. Module summary

By now it should be clear:

- the difference between `import x`, `import x as y`, and `from x import y`,
- how to create your own module (a `.py` file with functions) and import it,
- what `if __name__ == "__main__":` is for,
- what a package is, and what `__init__.py` does,
- how to install a third-party library (`pip install`) and save your dependencies
  (`requirements.txt`),
- why virtual environments exist and how to create one (`venv` or `conda`).

This also wraps up the practical side of organizing code in bigger projects — from here,
you can reach for any library on PyPI, knowing how to install it safely and keep it
isolated from other projects.

<a id="sec11"></a>
## 11. Exercises

Some tasks are practical use of standard modules, others involve creating your own
`.py` files with `%%writefile`, exactly like in the theory sections.

> 📝 **Exercise 1: `math` in practice**
>
> Import the `math` module. For `x = 47.3`, print: the square root, rounded up, rounded down, and the value of `math.pi` rounded to 3 decimal places.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import math

x = 47.3
print(math.sqrt(x))
print(math.ceil(x))
print(math.floor(x))
print(round(math.pi, 3))
```
</details>

> 📝 **Exercise 2: `random` in practice**
>
> Import `random`. Pick a random integer between 1 and 100, pick one random element from the list `["heads", "tails"]`, and shuffle (`.shuffle()`) the list `[1, 2, 3, 4, 5]`, printing it after shuffling.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import random

print(random.randint(1, 100))
print(random.choice(["heads", "tails"]))

numbers = [1, 2, 3, 4, 5]
random.shuffle(numbers)
print(numbers)
```
</details>

> 📝 **Exercise 3: Days until the holidays**
>
> Using `datetime`, compute how many days remain from today (`datetime.now()`) until a specific future date, e.g. `datetime(2026, 12, 25)` (Christmas). Hint: subtracting two `datetime` objects gives you a `timedelta`, which has a `.days` attribute.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
from datetime import datetime

now = datetime.now()
christmas = datetime(2026, 12, 25)

difference = christmas - now
print(f"Days until Christmas: {difference.days}")
```
</details>

> 📝 **Exercise 4: A custom currency converter module**
>
> Use `%%writefile` to create a file `currency.py` with a function `usd_to_eur(amount, rate=0.92)`, returning the amount multiplied by the rate. In the next cell, import that module and convert $500 to euros.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# Cell 1:
# %%writefile currency.py
# def usd_to_eur(amount, rate=0.92):
#     return amount * rate

# Cell 2:
# import currency
# print(currency.usd_to_eur(500))
```

This exercise needs two cells (the `%%writefile` magic has to be the only content of its cell) - the solution above shows both, but in practice split them into two separate code cells, just like in section 3.
</details>

> 📝 **Exercise 5: `__name__ == "__main__"` in action**
>
> Add an `if __name__ == "__main__":` block to the `currency.py` file from the previous exercise (or create it fresh) that prints the result of converting $1000 to euros. Confirm that `import currency` does NOT run that block, while `!python currency.py` does.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile currency.py
# def usd_to_eur(amount, rate=0.92):
#     return amount * rate
#
# if __name__ == "__main__":
#     print(f"$1000 is {usd_to_eur(1000):.2f} euros")

# import currency          # nothing extra gets printed
# !python currency.py       # this prints the conversion result
```

Hint: if the `currency` module was already imported once in this Jupyter session, importing it again does nothing (Python caches modules) - to see changes after editing the file, use `importlib.reload(currency)` or restart the kernel.
</details>

> 📝 **Exercise 6: Writing and reading JSON**
>
> Create a dictionary with data about yourself (name, age, favorite programming languages as a list). Save it to a file `profile.json` with `json.dump()`, then read it back with `json.load()` and check the `type()` of the loaded value.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import json

profile = {
    "name": "Kamil",
    "age": 30,
    "languages": ["Python", "SQL", "JavaScript"],
}

with open("profile.json", "w", encoding="utf-8") as file:
    json.dump(profile, file, indent=2)

with open("profile.json", "r", encoding="utf-8") as file:
    loaded_profile = json.load(file)

print(loaded_profile)
print(type(loaded_profile))
```
</details>

> 📝 **Exercise 7: A requirements.txt file**
>
> Using `%%writefile` (or plain `open()`/`.write()`), create a `requirements.txt` file for a project that uses: `pandas` at exactly version `2.2.0`, `requests` at any version, and `numpy` at version `1.24` or newer (the `>=` operator).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write("pandas==2.2.0\n")
    file.write("requests\n")
    file.write("numpy>=1.24\n")

with open("requirements.txt", "r", encoding="utf-8") as file:
    print(file.read())
```

In a real project, this file gets recreated with `pip install -r requirements.txt`, ideally inside an activated virtual environment.
</details>

> 🔥 **Exercise 8 (challenge): Your own mini-package**
>
> Create a package `my_tools` (a folder + `__init__.py`) with two modules: `text.py` with a function `is_palindrome(s)` (checks whether a string reads the same backward, after lowercasing), and `numbers.py` with a function `is_prime(n)` (checks whether a number is prime). Import both modules and test both functions on a few examples.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# Cell 1:
# import os
# os.makedirs("my_tools", exist_ok=True)

# Cell 2:
# %%writefile my_tools/__init__.py
# # empty file marking this as a package

# Cell 3:
# %%writefile my_tools/text.py
# def is_palindrome(s):
#     s = s.lower()
#     return s == s[::-1]

# Cell 4:
# %%writefile my_tools/numbers.py
# def is_prime(n):
#     if n < 2:
#         return False
#     for divisor in range(2, int(n ** 0.5) + 1):
#         if n % divisor == 0:
#             return False
#     return True

# Cell 5:
# from my_tools import text, numbers
#
# print(text.is_palindrome("kayak"))
# print(text.is_palindrome("Python"))
# print(numbers.is_prime(17))
# print(numbers.is_prime(18))
```

Hint: every `%%writefile` needs to be the only content of its cell - in a real notebook this is 5 separate cells run in order, exactly like the `text` module example in section 5.
</details>

---

### What's next?

Congratulations — that's the end of seven modules: from `print()` to organizing code
into modules, packages, and isolated environments. You now have the full toolkit to
start a real project, or reach for any library on PyPI (like `pandas` for data
analysis, `requests` for working with APIs, or `matplotlib` for charts) and know how to
install it safely.